In [6]:
# Testing NSE histories

import pandas as pd
from nse import nsefetch
import datetime
import requests
import logging


In [2]:
# Splitting datetime chunks

# Parameters
eq_symbol = 'PNB'
series = 'EQ'
days = 365
chunks = 35

end = datetime.datetime.now().date()
start = end-datetime.timedelta(days=days)

# convert datetimes to text for equities
eq_end = end.strftime('%d-%m-%Y')
eq_start = start.strftime('%d-%m-%Y')

periods = int((end-start).days/chunks)+1

bins = pd.date_range(start = start, end=end, periods=periods)
bins = pd.DatetimeIndex.strftime(bins, '%d-%m-%Y').astype(str)

# Generate urls based on split dates
eq_base = 'https://www.nseindia.com/api/historical/cm/equity?symbol='+eq_symbol
series_base = '&series='+'[%22'+series+'%22]'
dates_base = '&from='+eq_start+'&to='+eq_end
date_bases = [['&from='+eq_start+'&to='+eq_end ] for eq_start, eq_end in zip(bins[:-1], bins[1:])]
urls = [''.join([eq_base]+[series_base]+d) for d in date_bases]

In [3]:
# check one url (blocking)

url = urls[0]
payload = nsefetch(url)
df = pd.DataFrame.from_records(payload['data'])
df

,_id,CH_SYMBOL,CH_SERIES,CH_MARKET_TYPE,CH_TRADE_HIGH_PRICE,CH_TRADE_LOW_PRICE,CH_OPENING_PRICE,CH_CLOSING_PRICE,CH_LAST_TRADED_PRICE,CH_PREVIOUS_CLS_PRICE,...,CH_ISIN,CH_TIMESTAMP,TIMESTAMP,createdAt,updatedAt,__v,SLBMH_TOT_VAL,VWAP,mTIMESTAMP,CA
0,63ce8fc6eb800e0007026bcc,PNB,EQ,N,30.95,30.10,30.70,30.40,30.35,30.80,...,INE160A01022,2022-07-14,2022-07-13T18:30:00.000Z,2023-01-23T13:46:46.934Z,2023-01-23T13:46:46.934Z,0,None,30.45,14-Jul-2022,NaN
1,63ce8fb7c7c0c80007dbeccd,PNB,EQ,N,31.05,30.65,30.80,30.80,30.80,30.80,...,INE160A01022,2022-07-13,2022-07-12T18:30:00.000Z,2023-01-23T13:46:31.956Z,2023-01-23T13:46:31.956Z,0,None,30.82,13-Jul-2022,NaN
2,63ce8fa862e219000737c46f,PNB,EQ,N,31.20,30.65,30.80,30.80,30.65,31.05,...,INE160A01022,2022-07-12,2022-07-11T18:30:00.000Z,2023-01-23T13:46:16.798Z,2023-01-23T13:46:16.798Z,0,None,30.92,12-Jul-2022,NaN
3,63ce8f99a6943c000749d57e,PNB,EQ,N,31.15,30.50,30.50,31.05,30.95,30.65,...,INE160A01022,2022-07-11,2022-07-10T18:30:00.000Z,2023-01-23T13:46:01.991Z,2023-01-23T13:46:01.991Z,0,None,30.92,11-Jul-2022,NaN
4,63ce8f8aa6943c000749cbe3,PNB,EQ,N,31.20,30.40,31.10,30.65,30.65,31.00,...,INE160A01022,2022-07-08,2022-07-07T18:30:00.000Z,2023-01-23T13:45:47.009Z,2023-01-23T13:45:47.009Z,0,None,30.64,08-Jul-2022,NaN
5,63ce8f7b8e4da900077af575,PNB,EQ,N,31.10,30.00,30.10,31.00,30.95,29.95,...,INE160A01022,2022-07-07,2022-07-06T18:30:00.000Z,2023-01-23T13:45:31.445Z,2023-01-23T13:45:31.445Z,0,None,30.72,07-Jul-2022,NaN
6,63ce8f6ced0f3900079be5a6,PNB,EQ,N,30.05,29.55,29.60,29.95,29.95,29.70,...,INE160A01022,2022-07-06,2022-07-05T18:30:00.000Z,2023-01-23T13:45:16.334Z,2023-01-23T13:45:16.334Z,0,None,29.82,06-Jul-2022,NaN
7,63ce8f5dbca9320007ec08af,PNB,EQ,N,30.25,29.60,30.05,29.70,29.65,30.00,...,INE160A01022,2022-07-05,2022-07-04T18:30:00.000Z,2023-01-23T13:45:01.286Z,2023-01-23T13:45:01.286Z,0,None,29.95,05-Jul-2022,NaN
8,63ce8f4ec7c0c80007dbe323,PNB,EQ,N,30.10,29.45,29.45,30.00,30.00,29.45,...,INE160A01022,2022-07-04,2022-07-03T18:30:00.000Z,2023-01-23T13:44:46.072Z,2023-01-23T13:44:46.072Z,0,None,29.82,04-Jul-2022,NaN
9,63ce8f3f3853d8000737fd05,PNB,EQ,N,29.60,28.75,29.05,29.45,29.55,29.00,...,INE160A01022,2022-07-01,2022-06-30T18:30:00.000Z,2023-01-23T13:44:31.491Z,2023-01-23T13:44:31.491Z,0,None,29.17,01-Jul-2022,NaN


In [4]:
# import nest_asyncio
# nest_asyncio.apply()

url = urls[0]


In [15]:
# Fetch a payload asynchronously

import aiohttp
import asyncio

timeout_seconds = 2

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:74.0) Gecko/20100101 Firefox/74.0'
}

async def fetch(session, url, timeout_seconds):
    async with session.get(url, timeout = timeout_seconds) as resp:
        try:
            assert resp.status == 200
        except AssertionError:
            logging.error(AssertionError)
            s = await aiohttp.ClientSession(headers=headers, timeout=timeout_seconds)
            print(s)

        return await resp.text()

async def main(url, timeout_seconds):
    async with aiohttp.ClientSession(headers=headers, timeout=timeout_seconds) as session:
        html = await fetch(session, url, timeout_seconds)
        print(html)

# loop = asyncio.get_event_loop()
# loop.run_until_complete(main())
# asyncio.run(main())
await main(url, timeout_seconds)

ERROR:root:<class 'AssertionError'>
ERROR:asyncio:Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x000001C39262B700>


TypeError: object ClientSession can't be used in 'await' expression

In [13]:
# Fetch multiple payloads asynchronously

max_concurrent_requests = 2  # Maximum number of concurrent requests
timeout_seconds = 10  # Timeout for each individual request in seconds
throttle_delay = 1  # Delay between requests in seconds

async def fetch_with_throttling(client, url, throttle_delay):
    await asyncio.sleep(throttle_delay)  # Throttle requests
    return await fetch(client, url)

async def main(urls, timeout_seconds, max_concurrent_requests, throttle_delay):
    for url in urls:
        async with aiohttp.ClientSession(headers=headers, timeout=timeout_seconds) as session:
            semaphore = asyncio.Semaphore(max_concurrent_requests)
            async with semaphore:
                html = await fetch(session, url, timeout_seconds)
                print(html)        


In [14]:
await main(urls, timeout_seconds, max_concurrent_requests, throttle_delay)

{"data":[{"_id":"63ce8fc6eb800e0007026bcc","CH_SYMBOL":"PNB","CH_SERIES":"EQ","CH_MARKET_TYPE":"N","CH_TRADE_HIGH_PRICE":30.95,"CH_TRADE_LOW_PRICE":30.1,"CH_OPENING_PRICE":30.7,"CH_CLOSING_PRICE":30.4,"CH_LAST_TRADED_PRICE":30.35,"CH_PREVIOUS_CLS_PRICE":30.8,"CH_TOT_TRADED_QTY":16034640,"CH_TOT_TRADED_VAL":488182938.3,"CH_52WEEK_HIGH_PRICE":48.2,"CH_52WEEK_LOW_PRICE":28.05,"CH_TOTAL_TRADES":21146,"CH_ISIN":"INE160A01022","CH_TIMESTAMP":"2022-07-14","TIMESTAMP":"2022-07-13T18:30:00.000Z","createdAt":"2023-01-23T13:46:46.934Z","updatedAt":"2023-01-23T13:46:46.934Z","__v":0,"SLBMH_TOT_VAL":null,"VWAP":30.45,"mTIMESTAMP":"14-Jul-2022"},{"_id":"63ce8fb7c7c0c80007dbeccd","CH_SYMBOL":"PNB","CH_SERIES":"EQ","CH_MARKET_TYPE":"N","CH_TRADE_HIGH_PRICE":31.05,"CH_TRADE_LOW_PRICE":30.65,"CH_OPENING_PRICE":30.8,"CH_CLOSING_PRICE":30.8,"CH_LAST_TRADED_PRICE":30.8,"CH_PREVIOUS_CLS_PRICE":30.8,"CH_TOT_TRADED_QTY":14450589,"CH_TOT_TRADED_VAL":445373082.4,"CH_52WEEK_HIGH_PRICE":48.2,"CH_52WEEK_LOW_PRICE"

ERROR:root:<class 'AssertionError'>
C:\Users\kashi\AppData\Local\Temp\ipykernel_44768\3386757569.py:18: RuntimeWarning: coroutine 'ClientSession.close' was never awaited
  session.close()
ERROR:asyncio:Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x000001C3920CD030>


TypeError: object ClientSession can't be used in 'await' expression

In [ ]:
import asyncio
import aiohttp
import time

MAX_CONCURRENT_REQUESTS = 2  # Maximum number of concurrent requests
REQUEST_TIMEOUT = 10  # Timeout for each individual request in seconds
THROTTLE_DELAY = 1  # Delay between requests in seconds

async def fetch(session, url):
    async with session.get(url, timeout=REQUEST_TIMEOUT) as response:
        return await response.text()

async def fetch_with_throttling(session, url):
    await asyncio.sleep(THROTTLE_DELAY)  # Throttle requests
    return await fetch(session, url)

async def main(urls):
    async with aiohttp.ClientSession() as session:
        tasks = []
        semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)
        
        for url in urls:
            async with semaphore:
                task = asyncio.ensure_future(fetch_with_throttling(session, url))
                tasks.append(task)

        responses = await asyncio.gather(*tasks, return_exceptions=True)
        
        # Process the responses
        for url, response in zip(urls, responses):
            if isinstance(response, Exception):
                print(f"Error fetching URL {url}: {response}")
            else:
                print(f"Response from {url}: {response[:50]}...")
    
if __name__ == '__main__':
    start_time = time.time()
    loop = asyncio.get_event_loop()
    loop.run_until_complete(main(urls))
    elapsed_time = time.time() - start_time
    print(f"Total elapsed time: {elapsed_time:.2f} seconds")


In [ ]:
import asyncio
import aiohttp

coros = []

async def async_fetch(url):

    output = nsefetch(url)

    return output

my_timeout = aiohttp.ClientTimeout(
    total=None, # default value is 5 minutes, set to `None` for unlimited timeout
    sock_connect=4, # How long to wait before an open socket allowed to connect
    sock_read=4 # How long to wait with no data being read before timing out
)

client_args = dict(
    trust_env=True,
    timeout=my_timeout
)

# Pass args
async with aiohttp.ClientSession(**client_args) as session:
    ...
    # do stuff here
    for url in urls:
        coros.append(async_fetch(url))
    
    # Try awaiting, if a timeout, do something else
    try:
        values = await asyncio.gather(*coros, return_exceptions=True)
    except asyncio.exceptions.TimeoutError:
        print('Timeout!')

In [ ]:
values

In [ ]:
import grequests